###Tạo dữ liệu (DataFrame)

In [ ]:
import pandas as pd

# Bảng student
student = pd.DataFrame({
    "student_id": [1,2,3,4,5,6,7,8,9,10],
    "name": ["Nguyen Minh Hoang","Tran Thi Lan","Pham Van Nam","Le Thanh Huyen",
             "Vu Quoc Anh","Dang Thuy Linh","Bui Tien Dung","Ho Ngoc Mai",
             "Duong Huu Phuc","Cao Thi Hanh"],
    "class": ["May Tinh","Kinh Te","Toan Tin","Toan Tin","May Tinh",
              "May Tinh","Kinh Te","Toan Tin","Kinh Te","May Tinh"],
    "course_id": [12,34,None,20,24,24,34,20,None,None],
    "score": [6.7,9.2,7.9,7.2,8.0,5.5,9.2,8.8,7.2,7.0]
})

# Bảng course
course = pd.DataFrame({
    "id": [12,34,26],
    "course_name": ["Giai tich","Thong ke","Tin hoc"]
})

print("Student:")
print(student)
print("\nCourse:")
print(course)

Student:
   student_id               name     class  course_id  score
0           1  Nguyen Minh Hoang  May Tinh       12.0    6.7
1           2       Tran Thi Lan   Kinh Te       34.0    9.2
2           3       Pham Van Nam  Toan Tin        NaN    7.9
3           4     Le Thanh Huyen  Toan Tin       20.0    7.2
4           5        Vu Quoc Anh  May Tinh       24.0    8.0
5           6     Dang Thuy Linh  May Tinh       24.0    5.5
6           7      Bui Tien Dung   Kinh Te       34.0    9.2
7           8        Ho Ngoc Mai  Toan Tin       20.0    8.8
8           9     Duong Huu Phuc   Kinh Te        NaN    7.2
9          10       Cao Thi Hanh  May Tinh        NaN    7.0

Course:
   id course_name
0  12   Giai tich
1  34    Thong ke
2  26     Tin hoc


# Nhận xét
- Nhận xét bảng Student
- Ưu điểm
Có đầy đủ các thuộc tính quan trọng:
student_id: định danh duy nhất
name: tên sinh viên
class: lớp
course_id: mã môn học
score: điểm số
Dữ liệu điểm số nằm trong khoảng hợp lý (5.5 → 9.2)
- Vấn đề của bảng student là :     
Thiếu dữ liệu (NaN):
course_id bị thiếu ở các sinh viên:
Phạm Văn Nam
Dương Hữu Phúc
Cao Thị Hạnh
→ Gây khó khăn khi JOIN với bảng Course
Trùng giá trị course_id:
Nhiều sinh viên học cùng 1 môn (ví dụ: 34, 24)
→ Đây là điều bình thường (quan hệ nhiều–1)
course_id = 24 không tồn tại trong bảng Course
→ Dữ liệu bị lệch khóa ngoại (foreign key mismatch)
- Nhận xét bảng Course
- Ưu điểm
Cấu trúc đơn giản, rõ ràng:
id: mã môn
course_name: tên môn
Không có dữ liệu bị thiếu
- Vấn đề
Chỉ có 3 môn học
Thiếu các mã môn đang có trong Student:
Không có id = 20
Không có id = 24

→ Điều này gây lỗi khi JOIN (mất dữ liệu hoặc NULL)

In [ ]:

import sqlite3

conn = sqlite3.connect(':memory:')

# Đưa lại data vào SQL (nếu chưa)
student.to_sql('student', conn, index=False, if_exists='replace')
course.to_sql('course', conn, index=False, if_exists='replace')

3

###Câu 1

### Sử dụng Tích Descartes

In [ ]:
cartesian = student.merge(course, how='cross')
print(cartesian)

    student_id               name     class  course_id  score  id course_name
0            1  Nguyen Minh Hoang  May Tinh       12.0    6.7  12   Giai tich
1            1  Nguyen Minh Hoang  May Tinh       12.0    6.7  34    Thong ke
2            1  Nguyen Minh Hoang  May Tinh       12.0    6.7  26     Tin hoc
3            2       Tran Thi Lan   Kinh Te       34.0    9.2  12   Giai tich
4            2       Tran Thi Lan   Kinh Te       34.0    9.2  34    Thong ke
5            2       Tran Thi Lan   Kinh Te       34.0    9.2  26     Tin hoc
6            3       Pham Van Nam  Toan Tin        NaN    7.9  12   Giai tich
7            3       Pham Van Nam  Toan Tin        NaN    7.9  34    Thong ke
8            3       Pham Van Nam  Toan Tin        NaN    7.9  26     Tin hoc
9            4     Le Thanh Huyen  Toan Tin       20.0    7.2  12   Giai tich
10           4     Le Thanh Huyen  Toan Tin       20.0    7.2  34    Thong ke
11           4     Le Thanh Huyen  Toan Tin       20.0    7.2  2

## Nhận xét
Phép tích Descartes (CROSS JOIN) tạo ra tất cả các tổ hợp giữa 2 bảng, dẫn đến:
- Dữ liệu bị nhân bản
- Mất ý nghĩa thực tế
- Kết quả không phản ánh đúng mối quan hệ giữa sinh viên và môn học

### INNER JOIN

In [ ]:
inner_join = pd.merge(student, course,
                      left_on="course_id",
                      right_on="id",
                      how="inner")

print(inner_join)

   student_id               name     class  course_id  score  id course_name
0           1  Nguyen Minh Hoang  May Tinh       12.0    6.7  12   Giai tich
1           2       Tran Thi Lan   Kinh Te       34.0    9.2  34    Thong ke
2           7      Bui Tien Dung   Kinh Te       34.0    9.2  34    Thong ke


# Nhận xét  
Phép INNER JOIN giúp lọc ra các bản ghi có khóa khớp giữa hai bảng
Kết quả:
Chính xác, đúng logic
Nhưng làm mất các dữ liệu không hợp lệ hoặc bị thiếu

 Do đó:

-  Phù hợp khi cần độ chính xác cao
- Không phù hợp nếu muốn giữ toàn bộ dữ liệu

###LEFT JOIN

In [ ]:
left_join = pd.merge(student, course,
                     left_on="course_id",
                     right_on="id",
                     how="left")

print(left_join)

   student_id               name     class  course_id  score    id course_name
0           1  Nguyen Minh Hoang  May Tinh       12.0    6.7  12.0   Giai tich
1           2       Tran Thi Lan   Kinh Te       34.0    9.2  34.0    Thong ke
2           3       Pham Van Nam  Toan Tin        NaN    7.9   NaN         NaN
3           4     Le Thanh Huyen  Toan Tin       20.0    7.2   NaN         NaN
4           5        Vu Quoc Anh  May Tinh       24.0    8.0   NaN         NaN
5           6     Dang Thuy Linh  May Tinh       24.0    5.5   NaN         NaN
6           7      Bui Tien Dung   Kinh Te       34.0    9.2  34.0    Thong ke
7           8        Ho Ngoc Mai  Toan Tin       20.0    8.8   NaN         NaN
8           9     Duong Huu Phuc   Kinh Te        NaN    7.2   NaN         NaN
9          10       Cao Thi Hanh  May Tinh        NaN    7.0   NaN         NaN


# Nhận xét
Phép LEFT JOIN giữ lại toàn bộ dữ liệu từ bảng chính (Student)
Cho phép:
- Hiển thị đầy đủ dữ liệu
- Phát hiện các giá trị thiếu hoặc sai

 Tuy nhiên:

- Kết quả có thể chứa nhiều giá trị NULL
- Cần xử lý thêm trước khi phân tích

####RIGHT JOIN

In [ ]:
right_join = pd.merge(student, course,
                      left_on="course_id",
                      right_on="id",
                      how="right")

print(right_join)

   student_id               name     class  course_id  score  id course_name
0         1.0  Nguyen Minh Hoang  May Tinh       12.0    6.7  12   Giai tich
1         2.0       Tran Thi Lan   Kinh Te       34.0    9.2  34    Thong ke
2         7.0      Bui Tien Dung   Kinh Te       34.0    9.2  34    Thong ke
3         NaN                NaN       NaN        NaN    NaN  26     Tin hoc


# Nhận xét
Phép RIGHT JOIN giữ toàn bộ dữ liệu của bảng Course
Giúp:
- Phân tích tình trạng môn học (có sinh viên hay không)

 Tuy nhiên:

- Làm mất dữ liệu sinh viên không hợp lệ
- Xuất hiện giá trị NULL ở phía Student

###FULL OUTER JOIN

In [ ]:
full_join = pd.merge(student, course,
                     left_on="course_id",
                     right_on="id",
                     how="outer")

print(full_join)

    student_id               name     class  course_id  score    id  \
0          1.0  Nguyen Minh Hoang  May Tinh       12.0    6.7  12.0   
1          4.0     Le Thanh Huyen  Toan Tin       20.0    7.2   NaN   
2          8.0        Ho Ngoc Mai  Toan Tin       20.0    8.8   NaN   
3          5.0        Vu Quoc Anh  May Tinh       24.0    8.0   NaN   
4          6.0     Dang Thuy Linh  May Tinh       24.0    5.5   NaN   
5          NaN                NaN       NaN        NaN    NaN  26.0   
6          2.0       Tran Thi Lan   Kinh Te       34.0    9.2  34.0   
7          7.0      Bui Tien Dung   Kinh Te       34.0    9.2  34.0   
8          3.0       Pham Van Nam  Toan Tin        NaN    7.9   NaN   
9          9.0     Duong Huu Phuc   Kinh Te        NaN    7.2   NaN   
10        10.0       Cao Thi Hanh  May Tinh        NaN    7.0   NaN   

   course_name  
0    Giai tich  
1          NaN  
2          NaN  
3          NaN  
4          NaN  
5      Tin hoc  
6     Thong ke  
7     Thong

##Nhận xét
Phép FULL OUTER JOIN giữ lại toàn bộ dữ liệu của cả hai bảng
Giúp phát hiện:
- Dữ liệu thiếu
- Dữ liệu sai
- Quan hệ không khớp

 Tuy nhiên:

Kết quả chứa nhiều giá trị NULL
cần xử lý trước khi phân tích

# Câu 2

##Xóa các bản ghi có course_id không hợp lệ

In [ ]:
conn.execute("""
DELETE FROM student
WHERE course_id IS NOT NULL
AND course_id NOT IN (SELECT id FROM course)
""")

#Thêm dữ liệu còn thiếu

In [ ]:
conn.execute("""
UPDATE student
SET course_id = (SELECT MIN(id) FROM course)
WHERE course_id IS NULL
""")

#Tổng số sinh viên, điểm trung bình của từng lớp.

In [ ]:
df_a = pd.read_sql_query("""
SELECT class,
       COUNT(*) AS tong_sinh_vien,
       ROUND(AVG(score), 2) AS diem_trung_binh
FROM student
GROUP BY class
""", conn)

df_a

,class,tong_sinh_vien,diem_trung_binh
0,Kinh Te,3,8.53
1,May Tinh,2,6.85
2,Toan Tin,1,7.90


# Nhận xét
- Lớp Kinh Tế có kết quả học tập tốt nhất cả về số lượng và chất lượng
- Lớp Máy Tính có điểm trung bình thấp hơn, cần cải thiện
- Lớp Toán Tin có ít dữ liệu nên chưa thể đánh giá chính xác

#Tổng số sinh viên + điểm trung bình theo môn học

In [ ]:
df_b = pd.read_sql_query("""
SELECT c.course_name,
       COUNT(*) AS tong_sinh_vien,
       ROUND(AVG(s.score), 2) AS diem_trung_binh
FROM student s
JOIN course c ON s.course_id = c.id
GROUP BY c.course_name
""", conn)

df_b

,course_name,tong_sinh_vien,diem_trung_binh
0,Giai tich,4,7.2
1,Thong ke,2,9.2


## Nhận xét
- Môn Thống kê có kết quả học tập tốt nhất với điểm trung bình cao
- Môn Giải tích có nhiều sinh viên hơn nhưng điểm trung bình thấp hơn

#Phân loại thi đua theo điểm trung bình

In [ ]:
df_c = pd.read_sql_query("""
SELECT c.course_name,
       COUNT(*) AS tong_sinh_vien,
       ROUND(AVG(s.score), 2) AS diem_trung_binh,
       CASE
           WHEN AVG(s.score) >= 9 THEN 'Xuat sac'
           WHEN AVG(s.score) >= 6 THEN 'Tot'
           ELSE 'Kem'
       END AS xep_loai
FROM student s
JOIN course c ON s.course_id = c.id
GROUP BY c.course_name
""", conn)

df_c

,course_name,tong_sinh_vien,diem_trung_binh,xep_loai
0,Giai tich,4,7.2,Tot
1,Thong ke,2,9.2,Xuat sac


# Nhận xét
Việc phân loại thi đua đã được thực hiện đúng theo ngưỡng điểm trung bình
- Môn Thống kê đạt danh hiệu Xuất sắc
- Môn Giải tích đạt mức Tốt
- Không có môn nào ở mức Khá hoặc Trung bình

# Câu 3

##Xếp hạng theo điểm toàn bộ sinh viên

In [ ]:
df_rank_all = pd.read_sql_query("""
SELECT s1.*,
       (
         SELECT COUNT(*) + 1
         FROM student s2
         WHERE s2.score > s1.score
       ) AS rank_score
FROM student s1
ORDER BY rank_score
""", conn)

df_rank_all

,student_id,name,class,course_id,score,rank_score
0,2,Tran Thi Lan,Kinh Te,34.0,9.2,1
1,7,Bui Tien Dung,Kinh Te,34.0,9.2,1
2,3,Pham Van Nam,Toan Tin,12.0,7.9,3
3,9,Duong Huu Phuc,Kinh Te,12.0,7.2,4
4,10,Cao Thi Hanh,May Tinh,12.0,7.0,5
5,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,6


# Nhận xét
Việc xếp hạng phản ánh chính xác sự chênh lệch điểm giữa các sinh viên
Nhóm sinh viên
- Kinh Tế chiếm vị trí cao nhất
- Có sự phân hóa rõ rệt giữa nhóm điểm cao và trung bình

# Top 3 cao nhất

In [ ]:
top3_high = pd.read_sql_query("""
SELECT *
FROM (
    SELECT s1.*,
           (
             SELECT COUNT(*) + 1
             FROM student s2
             WHERE s2.score > s1.score
           ) AS rank_score
    FROM student s1
)
WHERE rank_score <= 3
""", conn)

top3_high

,student_id,name,class,course_id,score,rank_score
0,2,Tran Thi Lan,Kinh Te,34.0,9.2,1
1,3,Pham Van Nam,Toan Tin,12.0,7.9,3
2,7,Bui Tien Dung,Kinh Te,34.0,9.2,1


# Nhận xét
- Hai sinh viên lớp Kinh Tế dẫn đầu với thành tích xuất sắc, thể hiện chất lượng học tập vượt trội
- Sinh viên xếp sau có điểm thấp hơn đáng kể, cho thấy sự phân hóa rõ ràng giữa các nhóm học lực
- Việc sử dụng hàm RANK() giúp phản ánh chính xác trường hợp đồng hạng và thứ tự thực tế

#Top 3 thấp nhất

In [ ]:
top3_low = pd.read_sql_query("""
SELECT *
FROM (
    SELECT s1.*,
           (
             SELECT COUNT(*) + 1
             FROM student s2
             WHERE s2.score < s1.score
           ) AS rank_score
    FROM student s1
)
WHERE rank_score <= 3
""", conn)

top3_low

,student_id,name,class,course_id,score,rank_score
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,1
1,9,Duong Huu Phuc,Kinh Te,12.0,7.2,3
2,10,Cao Thi Hanh,May Tinh,12.0,7.0,2


# Nhận xét
- Nhóm sinh viên có điểm thấp chủ yếu thuộc lớp Máy Tính
- Mức điểm không quá thấp nhưng vẫn cần cải thiện
- Sự chênh lệch điểm giữa các sinh viên trong nhóm này không lớn

#Xếp hạng theo lớp

In [ ]:
df_rank_class = pd.read_sql_query("""
SELECT s1.*,
       (
         SELECT COUNT(*) + 1
         FROM student s2
         WHERE s2.class = s1.class
           AND s2.score > s1.score
       ) AS rank_in_class
FROM student s1
ORDER BY class, rank_in_class
""", conn)

df_rank_class

,student_id,name,class,course_id,score,rank_in_class
0,2,Tran Thi Lan,Kinh Te,34.0,9.2,1
1,7,Bui Tien Dung,Kinh Te,34.0,9.2,1
2,9,Duong Huu Phuc,Kinh Te,12.0,7.2,3
3,10,Cao Thi Hanh,May Tinh,12.0,7.0,1
4,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,2
5,3,Pham Van Nam,Toan Tin,12.0,7.9,1


# Nhận xét
Việc xếp hạng theo lớp giúp đánh giá chính xác năng lực sinh viên trong từng nhóm
- Lớp Kinh Tế có kết quả nổi bật nhất, trong khi lớp Máy Tính có mức điểm thấp hơn
- Sự phân hóa học lực thể hiện rõ ở một số lớp, đặc biệt là lớp Kinh Tế

#Top 3 mỗi lớp

In [ ]:
top3_class_high = pd.read_sql_query("""
SELECT *
FROM (
    SELECT s1.*,
           (
             SELECT COUNT(*) + 1
             FROM student s2
             WHERE s2.class = s1.class
               AND s2.score > s1.score
           ) AS rank_in_class
    FROM student s1
)
WHERE rank_in_class <= 3
""", conn)

top3_class_high

,student_id,name,class,course_id,score,rank_in_class
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,2
1,2,Tran Thi Lan,Kinh Te,34.0,9.2,1
2,3,Pham Van Nam,Toan Tin,12.0,7.9,1
3,7,Bui Tien Dung,Kinh Te,34.0,9.2,1
4,9,Duong Huu Phuc,Kinh Te,12.0,7.2,3
5,10,Cao Thi Hanh,May Tinh,12.0,7.0,1


# Nhận xét
- Việc lấy Top theo từng lớp giúp đánh giá chính xác sinh viên nổi bật trong từng nhóm
- Lớp Kinh Tế có chất lượng học tập tốt nhất
Các lớp còn lại chưa có sự nổi bật rõ ràng

#Top 3 thấp mỗi lớp

In [ ]:
pd.read_sql_query("""
SELECT *
FROM (
    SELECT s1.*,
           (
             SELECT COUNT(*) + 1
             FROM student s2
             WHERE s2.class = s1.class
               AND s2.score < s1.score
           ) AS rank_in_class
    FROM student s1
)
WHERE rank_in_class <= 3
""", conn)

,student_id,name,class,course_id,score,rank_in_class
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,1
1,2,Tran Thi Lan,Kinh Te,34.0,9.2,2
2,3,Pham Van Nam,Toan Tin,12.0,7.9,1
3,7,Bui Tien Dung,Kinh Te,34.0,9.2,2
4,9,Duong Huu Phuc,Kinh Te,12.0,7.2,1
5,10,Cao Thi Hanh,May Tinh,12.0,7.0,2


# Nhận xét
- Nhóm sinh viên có điểm thấp chủ yếu thuộc lớp Máy Tính
- Lớp Kinh Tế vẫn duy trì mức điểm khá trở lên dù xét ở nhóm thấp
- Việc xét top thấp theo lớp giúp xác định nhóm cần cải thiện trong từng lớp

#XẾP HẠNG THEO MÔN

In [ ]:
pd.read_sql_query("""
SELECT s1.*,
       (
         SELECT COUNT(*) + 1
         FROM student s2
         WHERE s2.course_id = s1.course_id
           AND s2.score > s1.score
       ) AS rank_in_course
FROM student s1
ORDER BY course_id, rank_in_course
""", conn)

,student_id,name,class,course_id,score,rank_in_course
0,3,Pham Van Nam,Toan Tin,12.0,7.9,1
1,9,Duong Huu Phuc,Kinh Te,12.0,7.2,2
2,10,Cao Thi Hanh,May Tinh,12.0,7.0,3
3,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,4
4,2,Tran Thi Lan,Kinh Te,34.0,9.2,1
5,7,Bui Tien Dung,Kinh Te,34.0,9.2,1


# Nhận xét
- Việc xếp hạng theo môn giúp đánh giá năng lực sinh viên trong từng lĩnh vực cụ thể
- Môn Thống kê có chất lượng học tập cao hơn so với Giải tích
- Sự xuất hiện đồng hạng cho thấy nhiều sinh viên đạt cùng mức điểm cao

#Top 3 mỗi môn

In [ ]:
pd.read_sql_query("""
SELECT *
FROM (
    SELECT s1.*,
           (
             SELECT COUNT(*) + 1
             FROM student s2
             WHERE s2.course_id = s1.course_id
               AND s2.score > s1.score
           ) AS rank_in_course
    FROM student s1
)
WHERE rank_in_course <= 3
""", conn)

,student_id,name,class,course_id,score,rank_in_course
0,2,Tran Thi Lan,Kinh Te,34.0,9.2,1
1,3,Pham Van Nam,Toan Tin,12.0,7.9,1
2,7,Bui Tien Dung,Kinh Te,34.0,9.2,1
3,9,Duong Huu Phuc,Kinh Te,12.0,7.2,2
4,10,Cao Thi Hanh,May Tinh,12.0,7.0,3


# Nhận xét
- Việc lọc Top 3 theo từng môn giúp xác định nhóm sinh viên có thành tích tốt nhất
- Môn Thống kê có kết quả vượt trội hơn so với Giải tích
- Sự xuất hiện đồng hạng phản ánh nhiều sinh viên đạt cùng mức điểm cao

#Top 3 thấp mỗi môn

In [ ]:
pd.read_sql_query("""
SELECT *
FROM (
    SELECT s1.*,
           (
             SELECT COUNT(*) + 1
             FROM student s2
             WHERE s2.course_id = s1.course_id
               AND s2.score < s1.score
           ) AS rank_in_course
    FROM student s1
)
WHERE rank_in_course <= 3
""", conn)

,student_id,name,class,course_id,score,rank_in_course
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,1
1,2,Tran Thi Lan,Kinh Te,34.0,9.2,1
2,7,Bui Tien Dung,Kinh Te,34.0,9.2,1
3,9,Duong Huu Phuc,Kinh Te,12.0,7.2,3
4,10,Cao Thi Hanh,May Tinh,12.0,7.0,2


# Nhận xét
- Việc xét top thấp theo môn giúp xác định môn học có nhiều sinh viên gặp khó khăn
- Môn Giải tích có kết quả thấp hơn, cần được chú ý cải thiện
- Môn Thống kê duy trì kết quả cao và ổn định

# Câu 4

# xác định thời gian tốt nghiệp

## Thêm cột vào


In [ ]:
conn.execute("""
ALTER TABLE student
ADD COLUMN graduation_date TEXT
""")

## cập nhật dữ liệu sv theo thứ hạng

In [ ]:
conn.execute("""
UPDATE student
SET graduation_date = datetime('now',
    '+' || (
        SELECT COUNT(*) + 1
        FROM student s2
        WHERE s2.score > student.score
    ) || ' days'
)
""")

Kết quả

In [ ]:
import pandas as pd

pd.read_sql_query("SELECT * FROM student", conn)

,student_id,name,class,course_id,score,graduation_date
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,2026-04-09 00:58:09
1,2,Tran Thi Lan,Kinh Te,34.0,9.2,2026-04-04 00:58:09
2,3,Pham Van Nam,Toan Tin,12.0,7.9,2026-04-06 00:58:09
3,7,Bui Tien Dung,Kinh Te,34.0,9.2,2026-04-04 00:58:09
4,9,Duong Huu Phuc,Kinh Te,12.0,7.2,2026-04-07 00:58:09
5,10,Cao Thi Hanh,May Tinh,12.0,7.0,2026-04-08 00:58:09


# Nhận xét
 Nhóm tốt nghiệp sớm nhất
- Tran Thi Lan – 2026-04-04
- Bui Tien Dung – 2026-04-04

Đây là những sinh viên tốt nghiệp sớm nhất
Đồng thời đều có điểm rất cao (9.2)
→ Cho thấy mối liên hệ tích cực giữa học lực và tiến độ tốt nghiệp
 Nhóm tốt nghiệp trung bình
- Pham Van Nam – 2026-04-06
- Duong Huu Phuc – 2026-04-07

 Nhận xét:

Thời gian tốt nghiệp ở mức trung bình
Điểm số ở mức khá (7.2 – 7.9)
 Nhóm tốt nghiệp muộn
- Cao Thi Hanh – 2026-04-08
- Nguyen Minh Hoang – 2026-04-09

 Nhận xét:

Đây là nhóm tốt nghiệp muộn nhất
Điểm số thấp hơn (6.7 – 7.0)
→ Có xu hướng học lực thấp → tốt nghiệp muộn hơn